In [ ]:
# BGF FASTA inference (temperature-scaled)
#
# Bare-bones inference for a FASTA file of new, unlabeled sequences.
# Loads one model checkpoint, scores every sequence in the FASTA file at a
# manually-designated temperature, and writes a predicted_label + confidence
# CSV per sequence. There are no true labels here, so no accuracy is computed.

!pip install evaluate prettytable peft
!pip install -U torchao

from google.colab import drive
drive.mount('/content/drive')

import sys
# point this at whatever folder contains train.py
sys.path.insert(0, "/content/drive/MyDrive/bgf_v2")

import os
import re
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, DataCollatorWithPadding
from safetensors.torch import load_file

# pull in the custom classes
from train import ESM2LoRAForSequenceClassification, ESMConfig


def parse_fasta(fasta_path):
    """Minimal FASTA parser -> list of (seq_id, sequence) tuples.
    Handles multi-line sequences. seq_id is the header text up to the first
    whitespace after '>'."""
    records = []
    seq_id, seq_chunks = None, []
    with open(fasta_path) as f:
        for line in f:
            line = line.rstrip()
            if not line:
                continue
            if line.startswith(">"):
                if seq_id is not None:
                    records.append((seq_id, "".join(seq_chunks)))
                header = line[1:]
                seq_id = header.split()[0] if header.split() else header
                seq_chunks = []
            else:
                seq_chunks.append(line)
        if seq_id is not None:
            records.append((seq_id, "".join(seq_chunks)))
    return records


class FastaInferenceDataset(Dataset):
    """Tokenizes sequences from a FASTA file. No labels.
    Truncates any sequence longer than max_length so an outlier-length
    sequence can't blow up memory on its batch."""
    def __init__(self, fasta_path, tokenizer, max_length=2048):
        self.records = parse_fasta(fasta_path)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        _, sequence = self.records[idx]
        inputs = self.tokenizer(
            sequence,
            truncation=True,
            max_length=self.max_length,
        )
        return {
            'input_ids':      inputs['input_ids'],
            'attention_mask': inputs['attention_mask'],
        }

def remap_layernorm_keys(state_dict):
    new_state = {}
    for k, v in state_dict.items():
        new_k = (k.replace(".LayerNorm.gamma", ".LayerNorm.weight")
                  .replace(".LayerNorm.beta", ".LayerNorm.bias"))
        new_state[new_k] = v
    return new_state


def run_inference(model_path, fasta_path, label_map_path, temperature,
                   out_path, tokenizer, device, batch_size=16, max_length=2048):
    """Load a checkpoint, score every sequence in a FASTA file at a fixed
    temperature, and write a per-sequence predicted_label + confidence CSV."""
    with open(label_map_path, "r") as f:
        label_mapper = json.load(f)
    id_to_label = {v: k for k, v in label_mapper.items()}

    config = ESMConfig.from_pretrained(model_path)
    state = load_file(os.path.join(model_path, "model.safetensors"))
    state = remap_layernorm_keys(state)   # <-- new line

    n_keep = max(int(m.group(1)) for k in state
                 if (m := re.search(r"encoder\.layer\.(\d+)\.", k))) + 1

    model = ESM2LoRAForSequenceClassification(config)
    base = model.get_submodule("backbone.base_model.model")
    base.encoder.layer = nn.ModuleList(list(base.encoder.layer)[:n_keep])
    base.pooler = None

    inc = model.load_state_dict(state, strict=False)
    assert not inc.missing_keys and not inc.unexpected_keys, inc
    print(f"clean load: {n_keep} layers, no pooler")
    model.to(device)
    model.eval()

    dataset = FastaInferenceDataset(fasta_path, tokenizer, max_length=max_length)
    seq_ids   = [r[0] for r in dataset.records]
    sequences = [r[1] for r in dataset.records]
    n_truncated = sum(1 for s in sequences if len(s) > max_length)
    print(f"Loaded {len(dataset)} sequences from {fasta_path}"
          f" ({n_truncated} longer than max_length={max_length}, will be truncated)")

    collator = DataCollatorWithPadding(tokenizer=tokenizer, padding="longest", return_tensors="pt")
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=collator)

    all_pred, all_conf = [], []
    with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.bfloat16):
        for batch in tqdm(loader, desc="Inference", unit="batch"):
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            _, logits = model(input_ids=input_ids, attention_mask=attention_mask)
            scaled = logits.float() / temperature
            probs = F.softmax(scaled, dim=1)
            conf, pred = probs.max(dim=1)
            all_pred.append(pred.cpu())
            all_conf.append(conf.cpu())

    all_pred = torch.cat(all_pred)
    all_conf = torch.cat(all_conf)

    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    results_df = pd.DataFrame({
        "seq_id":          seq_ids,
        "sequence":        sequences,
        "predicted_label": [id_to_label[i] for i in all_pred.tolist()],
        "confidence":      all_conf.tolist(),
    })
    results_df.to_csv(out_path, index=False)
    print(f"Saved predictions to {out_path}")

    return results_df


if __name__ == "__main__":
    tokenizer = AutoTokenizer.from_pretrained("facebook/esm2_t6_8M_UR50D")
    device = "cuda:0" if torch.cuda.is_available() else "cpu"

    # --- fill these in ---
    MODEL_PATH  = "/content/drive/MyDrive/bgf_v2/models/bgf_150M_50_1ep/checkpoint-3712"
    FASTA_PATH  = "/content/drive/MyDrive/bgf_v2/data/mags/cleaned_MAGS_forbgf.fasta"
    LABEL_MAP   = "/content/drive/MyDrive/bgf_v2/data/cyc_id_50_label_map.json"
    TEMPERATURE = 0.9116  # value chosen from a prior calibration sweep
    OUT_PATH    = "/content/drive/MyDrive/bgf_v2/results/mags/cleaned_MAGS_forbgf_output_50.csv"
    MAX_LENGTH  = 1024  # truncate any sequence longer than this

    run_inference(
        model_path=MODEL_PATH,
        fasta_path=FASTA_PATH,
        label_map_path=LABEL_MAP,
        temperature=TEMPERATURE,
        out_path=OUT_PATH,
        tokenizer=tokenizer,
        device=device,
        max_length=MAX_LENGTH,
    )

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 108.0 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
Mounted at /content/drive


config.json:   0%|          | 0.00/775 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/486 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: facebook/esm2_t30_150M_UR50D
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


clean load: 15 layers, no pooler
Loaded 6282956 sequences from /content/drive/MyDrive/bgf_v2/data/mags/cleaned_MAGS_forbgf.fasta (58550 longer than max_length=1024, will be truncated)


Inference:  76%|███████▌  | 296958/392685 [5:36:29<1:50:03, 14.50batch/s]